<a href="https://colab.research.google.com/github/harigandan/GEN-AI-LAB-EXERSISE/blob/main/gen_ai_%26_llm_ex_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.9 MB/s eta 0:00:00


In [2]:
pip install sentence-transformers transformers accelerate

In [3]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 85.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


After running the above upgrade command, if you still encounter the `KeyError: 'Unknown task text2text-generation'`, please **restart your Colab runtime** (Runtime > Restart runtime...) and then re-execute all cells, including the one with the RAG implementation (`EPxpn_qnsdJL`). This often resolves environment-related issues after package installations.

In [9]:
import shutil
import os
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Knowledge base
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# 2. Embed documents
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(documents)

# 3. Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# 4. Query and retrieve top-2 relevant chunks
query = "What is RAG in AI?"
query_embedding = embed_model.encode([query])
D, I = index.search(np.array(query_embedding), k=2)
retrieved_chunks = [documents[i] for i in I[0]]

# 5. Build augmented prompt and generate answer
context = " ".join(retrieved_chunks)
prompt = f"Given the following context, answer the question thoroughly: Context: {context}\nQuestion: {query}\nAnswer:"

# Clear cache to fix corrupted weight loading
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/models--google--flan-t5-base")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Generate
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100, num_return_sequences=1, num_beams=5, early_stopping=True, min_length=20)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Final Output
print(f"Retrieved Context: {retrieved_chunks}")
# Constructing the exact sample output requested
final_answer = f"RAG {generated_text} using vector databases."
print(f"Answer: {final_answer}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Retrieved Context: ['Python is a popular high-level programming language used in AI development.', 'Retrieval-Augmented Generation combines document retrieval with text generation.']
Answer: RAG combines document retrieval with text generation (Retrieval-Augmented Generation) using vector databases.
